# 11 · Gradient Boosting: XGBoost, LightGBM y CatBoost

Para datos tabulares, gradient boosting es una de las familias más competitivas. Este lab parte desde la intuición de boosting y llega a implementaciones modernas.

## Objetivos
- Entender additive models y corrección secuencial de residuos/gradientes.
- Relacionar learning rate, número de árboles, profundidad y regularización.
- Comparar HistGradientBoosting, XGBoost, LightGBM y CatBoost.
- Usar early stopping.
- Entender diferencias de manejo de categorías y missing.
- Evitar overfitting al tunear.


## 1. Idea del boosting
Construimos una función aditiva:
$$F_M(x)=F_0(x)+\sum_{m=1}^M \eta h_m(x)$$
Cada nuevo árbol intenta reducir la pérdida actual siguiendo el gradiente negativo. `learning_rate` ($\eta$) pequeño suele requerir más árboles pero permite ajustes más suaves.

Bagging (Random Forest) entrena árboles en paralelo para bajar variance; boosting los encadena para reducir errores sistemáticos.


In [ ]:
!pip -q install xgboost lightgbm catboost
import numpy as np, pandas as pd, time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
SEED=42
X,y=load_breast_cancer(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)

## 2. XGBoost
XGBoost popularizó regularización explícita, shrinkage, subsampling, column sampling, manejo de sparse data y optimizaciones de árboles. Hiperparámetros clave:
- `n_estimators`, `learning_rate`;
- `max_depth`, `min_child_weight`;
- `subsample`, `colsample_bytree`;
- `reg_alpha` L1, `reg_lambda` L2;
- `gamma` para exigir mejora mínima antes de split.


## 3. LightGBM
Usa histogramas y crecimiento leaf-wise, por lo que puede ser extremadamente rápido en datasets grandes. `num_leaves`, `min_child_samples`, `max_depth`, feature/bagging fractions son claves. Leaf-wise puede overfit datasets pequeños si se deja demasiada complejidad.

## 4. CatBoost
Diseñado especialmente para variables categóricas, con ordered target statistics para reducir leakage. Puede manejar categorías directamente y suele requerir menos preprocessing en problemas mixtos.


In [ ]:
models={
 'sklearn_hist':HistGradientBoostingClassifier(max_iter=300,learning_rate=.05,max_leaf_nodes=15,random_state=SEED),
 'xgboost':XGBClassifier(n_estimators=500,learning_rate=.03,max_depth=4,subsample=.85,colsample_bytree=.85,reg_lambda=2,eval_metric='logloss',random_state=SEED,n_jobs=-1),
 'lightgbm':LGBMClassifier(n_estimators=500,learning_rate=.03,num_leaves=15,max_depth=-1,subsample=.85,colsample_bytree=.85,reg_lambda=2,random_state=SEED,verbosity=-1,n_jobs=-1),
 'catboost':CatBoostClassifier(iterations=500,learning_rate=.03,depth=5,l2_leaf_reg=3,verbose=False,random_seed=SEED)
}
rows=[]
for name,m in models.items():
 t=time.time(); m.fit(Xtr,ytr); sec=time.time()-t; p=m.predict_proba(Xte)[:,1]
 rows.append([name,sec,roc_auc_score(yte,p),average_precision_score(yte,p)])
pd.DataFrame(rows,columns=['modelo','fit_sec','ROC_AUC','PR_AUC']).sort_values('PR_AUC',ascending=False).round(4)

## 5. Early stopping
En vez de fijar arbitrariamente 3000 árboles, observa una validation set y detén cuando la métrica deja de mejorar. Ojo: el conjunto usado para early stopping ya es validation y no debe ser el test final.


In [ ]:
Xtrain,Xval,ytrain,yval=train_test_split(Xtr,ytr,test_size=.2,stratify=ytr,random_state=SEED)
xgb=XGBClassifier(n_estimators=3000,learning_rate=.02,max_depth=4,subsample=.85,colsample_bytree=.85,eval_metric='logloss',early_stopping_rounds=80,random_state=SEED,n_jobs=-1)
xgb.fit(Xtrain,ytrain,eval_set=[(Xval,yval)],verbose=False)
print('best iteration',xgb.best_iteration,'test AUC',roc_auc_score(yte,xgb.predict_proba(Xte)[:,1]))

## 6. Categóricas con CatBoost
El one-hot puede explotar cardinalidad y target encoding ingenuo puede filtrar target. CatBoost implementa estadísticas ordenadas. En datasets con categorías de alta cardinalidad puede ser una primera opción fuerte.


In [ ]:
# ejemplo sintético mixto
rng=np.random.default_rng(SEED); n=5000
d=pd.DataFrame({'edad':rng.integers(18,85,n),'region':rng.choice(['N','C','S','A'],n),'ocupacion':rng.choice([f'cat_{i}' for i in range(30)],n)})
logit=-3+.035*d.edad+(d.region=='C')*.8+d.ocupacion.isin(['cat_2','cat_7','cat_9'])*1.2
d['y']=rng.binomial(1,1/(1+np.exp(-logit)))
tr,te=train_test_split(np.arange(n),test_size=.25,stratify=d.y,random_state=SEED)
cat_cols=['region','ocupacion']; m=CatBoostClassifier(iterations=400,depth=6,learning_rate=.05,verbose=False,random_seed=SEED)
m.fit(d.loc[tr,['edad','region','ocupacion']],d.loc[tr,'y'],cat_features=cat_cols)
print('AUC categórico',roc_auc_score(d.loc[te,'y'],m.predict_proba(d.loc[te,['edad','region','ocupacion']])[:,1]))

## 7. ¿Cuál elegir?
- **XGBoost:** ecosistema maduro, flexible y robusto.
- **LightGBM:** gran velocidad/escalabilidad en tabular grande.
- **CatBoost:** excelente con categóricas y menor ingeniería de encoding.
- **HistGradientBoosting:** sin dependencia externa y muy competitivo en sklearn.

La respuesta correcta es experimental: CV apropiada + costo de inferencia + mantenimiento.

## Errores comunes
- tunear cientos de combinaciones mirando test;
- profundidad excesiva con muchos árboles;
- learning rate alto + árboles profundos;
- ignorar imbalance/calibración;
- interpretar `gain` como causalidad;
- mezclar categorías codificadas arbitrariamente como enteros ordinales.

## Ejercicios
1. Grafica validation loss por iteración.
2. Compara `learning_rate` 0.3, 0.1, 0.03 y ajusta `n_estimators`.
3. Evalúa `subsample` y `colsample_bytree` como regularización.
4. Compara CatBoost categórico vs OneHot+LightGBM.
5. Usa Optuna para tuning.
6. Compara SHAP de los tres modelos.
7. Mide latencia por 1, 100 y 10.000 filas.
